# Running Models — Audio · Part 3 — Companion Notebook

This goes with **"Better Voices: Bark, MMS, and the Trade-offs"**. Run cells top to bottom (Shift+Enter).

This is a GPU-leaning notebook: **Runtime → Change runtime type → GPU** before running Bark, or it will crawl.

## Install

Both models ship through `transformers`.

In [ ]:
!pip install -q transformers

## Bark — the expressive one

From Suno. Does prosody and nonverbal sounds (laughs, sighs, even singing). We use `bark-small`, the lighter checkpoint. Bark really wants a GPU.

In [ ]:
from transformers import AutoProcessor, BarkModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained("suno/bark-small")
model = BarkModel.from_pretrained("suno/bark-small").to(device)

inputs = processor(
    "Whoa — this one actually sounds human. [laughs]",
    voice_preset="v2/en_speaker_6",
)
inputs = {k: v.to(device) for k, v in inputs.items()}
audio = model.generate(**inputs)

from IPython.display import Audio
Audio(audio.cpu().numpy().squeeze(), rate=model.generation_config.sample_rate)

### Tags and presets

Nonverbal tags go inline: `[laughs]`, `[sighs]`, `[gasps]`, `[clears throat]`, `[music]`. Wrap a line in `♪` and Bark sings it. Voices are fixed preset strings: `v2/en_speaker_0`..`v2/en_speaker_9`, plus other languages like `v2/de_speaker_3`.

In [ ]:
inputs = processor(
    "Singing now ♪ la la laaa ♪",
    voice_preset="v2/en_speaker_3",
)
inputs = {k: v.to(device) for k, v in inputs.items()}
audio = model.generate(**inputs)
Audio(audio.cpu().numpy().squeeze(), rate=model.generation_config.sample_rate)

## MMS — the multilingual one

Meta's MMS trained TTS for 1000+ languages on one architecture (VITS). Lean, fast, no GPU or speaker embedding needed. The language lives in the model id.

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch

model = VitsModel.from_pretrained("facebook/mms-tts-eng")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-eng")

inputs = tokenizer("One model, over a thousand languages.", return_tensors="pt")
with torch.no_grad():
    waveform = model(**inputs).waveform

Audio(waveform.numpy(), rate=model.config.sampling_rate)   # 16000

### Switch language by switching model

There's no `language=` argument — the ISO code in the model id is the whole interface. `mms-tts-deu` (German), `mms-tts-hin` (Hindi), `mms-tts-fra` (French), `mms-tts-spa` (Spanish), ...

In [ ]:
model = VitsModel.from_pretrained("facebook/mms-tts-deu")       # German
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-deu")

inputs = tokenizer("Ein Modell, über tausend Sprachen.", return_tensors="pt")
with torch.no_grad():
    waveform = model(**inputs).waveform

Audio(waveform.numpy(), rate=model.config.sampling_rate)

## The trade-offs

| | SpeechT5 (Part 2) | Bark | MMS / VITS |
|---|---|---|---|
| Expressiveness | Flat, robotic | High | Neutral, clean |
| Speed | Fast | Slow (wants GPU) | Fast |
| Compute | CPU is fine | GPU preferred | CPU is fine |
| Languages | English-focused | ~13 | 1000+ (one each) |
| Speaker embedding? | Required | No (presets) | No |

No winner row — pick the one that fits the job. Next up (Part 4): a narrator that reads a whole article aloud.